# Model Comparison

Train baseline models from near-original cleaned columns, compare validation metrics, and select the practical top 2 models for feature engineering.

In [7]:
from pathlib import Path
import time
import warnings

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

ROOT_DIR = Path.cwd().parents[1] if Path.cwd().name == "model_comparison" else Path.cwd()
DATA_PATH = ROOT_DIR / "data" / "diabetic_data_clean_common.csv"
OUTPUT_DIR = ROOT_DIR / "train" / "model_comparison" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMN = "readmitted_binary"
RANDOM_STATE = 42

DATA_PATH, OUTPUT_DIR

(WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/data/diabetic_data_clean_common.csv'),
 WindowsPath('d:/HocTap/KT&XLTT/CUOIKI/train/model_comparison/outputs'))

## Load Baseline Data

This baseline uses the near-original cleaned columns. The raw baseline has about 44 input columns before encoding. After one-hot encoding, the feature matrix becomes wider.

In [8]:
df = pd.read_csv(DATA_PATH)

DROP_COLUMNS = [
    "readmitted",
    "diag_1",
    "diag_2",
    "diag_3",
    "age",
    "age_midpoint",
]

y = df[TARGET_COLUMN]
X = df.drop(columns=[TARGET_COLUMN])
X = X.drop(columns=[col for col in DROP_COLUMNS if col in X.columns])

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=RANDOM_STATE,
)

categorical_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

print("Raw baseline X_train:", X_train_raw.shape)
print("Raw baseline X_val:", X_val_raw.shape)
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("y_train distribution:")
print(y_train.value_counts(normalize=True).rename("ratio"))

Raw baseline X_train: (65128, 44)
Raw baseline X_val: (16282, 44)
Numeric columns: 12
Categorical columns: 32
y_train distribution:
readmitted_binary
0    0.539108
1    0.460892
Name: ratio, dtype: float64


## Preprocess Baseline Columns

Fit scaler and one-hot encoder on train only, then transform validation. This keeps the baseline close to the original 44 columns while still making it usable for sklearn models.

In [9]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("onehot", make_one_hot_encoder())]), categorical_cols),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
feature_names = preprocessor.get_feature_names_out()

print("Processed X_train:", X_train.shape)
print("Processed X_val:", X_val.shape)
print("Feature count after encoding:", len(feature_names))

Processed X_train: (65128, 243)
Processed X_val: (16282, 243)
Feature count after encoding: 243


## Define Baseline Models

The models below are configured as practical baselines. `LinearSVC` is used for SVM because full kernel SVM can be very slow on this dataset size.

In [10]:
models = {
    "Dummy Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        solver="saga",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=10,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance",
        n_jobs=-1,
    ),
    "SVM": LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE,
    ),
    "Naive Bayes": GaussianNB(),
}

list(models.keys())

['Dummy Baseline',
 'Logistic Regression',
 'Decision Tree',
 'Random Forest',
 'KNN',
 'SVM',
 'Naive Bayes']

## Train And Evaluate

Metrics are computed on validation. Naive Bayes may get high F1 by predicting class 1 too often, so final ranking also considers Accuracy, Precision, ROC-AUC, and prediction balance.

In [11]:
def get_score_for_auc(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate_model(name, model, X_train, y_train, X_val, y_val):
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time

    start_time = time.time()
    y_pred = model.predict(X_val)
    predict_time = time.time() - start_time

    y_score = get_score_for_auc(model, X_val)
    roc_auc = roc_auc_score(y_val, y_score) if y_score is not None else None

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1_score": f1_score(y_val, y_pred, zero_division=0),
        "roc_auc": roc_auc,
        "train_time_sec": train_time,
        "predict_time_sec": predict_time,
    }

    cm = pd.DataFrame(
        confusion_matrix(y_val, y_pred),
        index=["actual_0", "actual_1"],
        columns=["predicted_0", "predicted_1"],
    )
    return metrics, cm, model


results = []
confusion_matrices = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    metrics, cm, trained_model = evaluate_model(name, model, X_train, y_train, X_val, y_val)
    results.append(metrics)
    confusion_matrices[name] = cm
    trained_models[name] = trained_model
    print(
        f"Done {name}: "
        f"F1={metrics['f1_score']:.4f}, "
        f"Recall={metrics['recall']:.4f}, "
        f"Precision={metrics['precision']:.4f}, "
        f"Accuracy={metrics['accuracy']:.4f}"
    )

comparison_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)
comparison_df

Training Dummy Baseline...
Done Dummy Baseline: F1=0.0000, Recall=0.0000, Precision=0.0000, Accuracy=0.5391
Training Logistic Regression...
Done Logistic Regression: F1=0.5837, Recall=0.5722, Precision=0.5956, Accuracy=0.6238
Training Decision Tree...
Done Decision Tree: F1=0.5807, Recall=0.5713, Precision=0.5905, Accuracy=0.6198
Training Random Forest...
Done Random Forest: F1=0.6144, Recall=0.6141, Precision=0.6148, Accuracy=0.6448
Training KNN...
Done KNN: F1=0.5409, Recall=0.5068, Precision=0.5798, Accuracy=0.6034
Training SVM...
Done SVM: F1=0.5856, Recall=0.5766, Precision=0.5949, Accuracy=0.6239
Training Naive Bayes...
Done Naive Bayes: F1=0.6322, Recall=0.9274, Precision=0.4796, Accuracy=0.5027


,model,accuracy,precision,recall,f1_score,roc_auc,train_time_sec,predict_time_sec
6,Naive Bayes,0.502702,0.479567,0.927372,0.632205,0.614538,0.254677,0.088973
3,Random Forest,0.644823,0.614810,0.614072,0.614441,0.699756,5.260080,0.135712
5,SVM,0.623941,0.594940,0.576626,0.585640,0.668293,6.912031,0.011194
1,Logistic Regression,0.623756,0.595562,0.572228,0.583662,0.669743,316.351902,0.006534
2,Decision Tree,0.619826,0.590496,0.571295,0.580737,0.669498,1.413650,0.009992
4,KNN,0.603427,0.579814,0.506796,0.540852,0.643586,0.011172,2.869848
0,Dummy Baseline,0.539123,0.000000,0.000000,0.000000,0.500000,0.001967,0.000141


## Practical Ranking And Top 2 Models

The practical ranking below follows the baseline result table. Naive Bayes is not selected as top model because its recall is extremely high but precision, accuracy, and ROC-AUC are poor, indicating biased predictions toward class 1.

In [12]:
rank_order = [
    "Random Forest",
    "SVM",
    "Logistic Regression",
    "Decision Tree",
    "Naive Bayes",
    "KNN",
    "Dummy Baseline",
]

assessment_map = {
    "Random Forest": "Tốt nhất tổng thể",
    "SVM": "Ổn định",
    "Logistic Regression": "Baseline tốt, dễ giải thích",
    "Decision Tree": "Recall tốt nhưng kém ổn định hơn",
    "Naive Bayes": "Recall rất cao nhưng dự đoán lệch",
    "KNN": "F1 thấp, predict chậm",
    "Dummy Baseline": "Mốc so sánh",
}

practical_ranking_df = comparison_df.copy()
practical_ranking_df["rank"] = practical_ranking_df["model"].map(
    {model_name: rank for rank, model_name in enumerate(rank_order, start=1)}
)
practical_ranking_df["assessment"] = practical_ranking_df["model"].map(assessment_map)
practical_ranking_df = practical_ranking_df.sort_values("rank").reset_index(drop=True)
practical_ranking_df = practical_ranking_df[
    ["rank", "model", "accuracy", "precision", "recall", "f1_score", "roc_auc", "assessment"]
]

top2_df = practical_ranking_df.head(2).reset_index(drop=True)
top2_models = top2_df["model"].tolist()

print("Practical ranking:")
display(practical_ranking_df)

print("Top 2 models selected for feature engineering:")
display(top2_df)

top2_models

Practical ranking:


,rank,model,accuracy,precision,recall,f1_score,roc_auc,assessment
0,1,Random Forest,0.644823,0.614810,0.614072,0.614441,0.699756,Tốt nhất tổng thể
1,2,SVM,0.623941,0.594940,0.576626,0.585640,0.668293,Ổn định
2,3,Logistic Regression,0.623756,0.595562,0.572228,0.583662,0.669743,"Baseline tốt, dễ giải thích"
3,4,Decision Tree,0.619826,0.590496,0.571295,0.580737,0.669498,Recall tốt nhưng kém ổn định hơn
4,5,Naive Bayes,0.502702,0.479567,0.927372,0.632205,0.614538,Recall rất cao nhưng dự đoán lệch
5,6,KNN,0.603427,0.579814,0.506796,0.540852,0.643586,"F1 thấp, predict chậm"
6,7,Dummy Baseline,0.539123,0.000000,0.000000,0.000000,0.500000,Mốc so sánh


Top 2 models selected for feature engineering:


,rank,model,accuracy,precision,recall,f1_score,roc_auc,assessment
0,1,Random Forest,0.644823,0.61481,0.614072,0.614441,0.699756,Tốt nhất tổng thể
1,2,SVM,0.623941,0.59494,0.576626,0.585640,0.668293,Ổn định


['Random Forest', 'SVM']

## Save Outputs

In [13]:
comparison_df.to_csv(OUTPUT_DIR / "model_comparison_metrics.csv", index=False)
practical_ranking_df.to_csv(OUTPUT_DIR / "practical_model_ranking.csv", index=False)
top2_df.to_csv(OUTPUT_DIR / "top2_models.csv", index=False)

for name, cm in confusion_matrices.items():
    safe_name = name.lower().replace(" ", "_")
    cm.to_csv(OUTPUT_DIR / f"{safe_name}_validation_confusion_matrix.csv")

for name in top2_models:
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(trained_models[name], OUTPUT_DIR / f"{safe_name}_model.joblib")

joblib.dump(preprocessor, OUTPUT_DIR / "baseline_preprocessor.joblib")
pd.Series(feature_names, name="feature_name").to_csv(OUTPUT_DIR / "baseline_encoded_feature_names.csv", index=False)

print(f"Saved outputs to: {OUTPUT_DIR}")

Saved outputs to: d:\HocTap\KT&XLTT\CUOIKI\train\model_comparison\outputs
